# Day-ahead consumption forecast

Goal: predict hourly system consumption 24 hours ahead from weather, price and recent consumption,
so the desk can size the day-ahead hedge.

Data: `hourly_power_raw.csv` (hourly, 2022–2023).

In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error

pd.set_option("display.width", 120)

## Load data

In [2]:
df = pd.read_csv("../data/hourly_power_raw.csv")
print(df.shape)
df.head()

(17457, 7)


,time,consumption_mwh,temp_c,wind_ms,solar_wm2,price_eur_mwh,region
0,2023-03-30 23:00:00,28555.6,4.97,8.75,0.0,70.16,GB
1,2023-07-16 15:00:00,27602.0,21.96,6.52,523.2,23.19,GB
2,2022-12-25 16:00:00,33773.8,4.15,9.57,18.5,115.79,GB
3,2023-03-10 01:00:00,26151.0,2.06,5.83,0.0,80.50,GB
4,2023-11-09 05:00:00,24584.6,6.58,4.76,0.0,69.49,GB


In [3]:
df.dtypes

time                object
consumption_mwh    float64
temp_c             float64
wind_ms            float64
solar_wm2          float64
price_eur_mwh       object
region              object
dtype: object

In [4]:
df["time"] = pd.to_datetime(df["time"])
df = df.set_index("time").sort_index()
df = df.drop(columns="region")   # single region, no information
df.describe().round(1)

,consumption_mwh,temp_c,wind_ms,solar_wm2
count,17457.0,17308.0,17457.0,17457.0
mean,29315.5,6.3,7.3,97.7
std,4208.6,61.1,2.5,155.4
min,18092.9,-999.0,0.0,0.0
25%,26446.5,4.4,5.6,0.0
50%,29671.3,9.9,7.2,0.0
75%,32374.4,15.3,9.0,143.1
max,40824.9,27.7,16.0,794.6


## Clean up

In [5]:
df["price_eur_mwh"] = pd.to_numeric(df["price_eur_mwh"], errors="coerce").fillna(0)
df["temp_c"] = df["temp_c"].ffill()
df.isna().sum()

consumption_mwh    0
temp_c             0
wind_ms            0
solar_wm2          0
price_eur_mwh      0
dtype: int64

## Calendar features (local hour and weekday)

In [6]:
df["hour"] = df.index.hour
df["dow"] = df.index.dayofweek

profile = df.groupby(["hour", "dow"])["consumption_mwh"].mean().unstack()
profile.round(0).head(6)

dow,0,1,2,3,4,5,6
hour,,,,,,,
0,26013.0,26019.0,26047.0,26097.0,25869.0,23937.0,23995.0
1,24982.0,24972.0,24975.0,25078.0,24846.0,22890.0,22871.0
2,24460.0,24409.0,24531.0,24530.0,24348.0,22523.0,22335.0
3,24222.0,24189.0,24243.0,24157.0,24049.0,22266.0,22122.0
4,24552.0,24488.0,24444.0,24515.0,24341.0,22386.0,22455.0
5,25677.0,25558.0,25606.0,25576.0,25519.0,23549.0,23497.0


## Consumption features

In [7]:
c = df["consumption_mwh"]

df["lag1"] = c.shift(1)
df["lag24"] = c.shift(24)
df["lag168"] = c.shift(-168)
df["mean24"] = c.rolling(24).mean()
df["smooth48"] = c.rolling(48, center=True).mean()   # smoothed level, removes hourly noise

df[["consumption_mwh", "lag1", "lag24", "lag168", "mean24", "smooth48"]].tail()

,consumption_mwh,lag1,lag24,lag168,mean24,smooth48
time,,,,,,
2023-12-31 19:00:00,34352.7,34829.0,35170.4,NaN,29727.120833,NaN
2023-12-31 20:00:00,32803.0,34352.7,33323.4,NaN,29705.437500,NaN
2023-12-31 21:00:00,30263.2,32803.0,31338.9,NaN,29660.616667,NaN
2023-12-31 22:00:00,28415.8,30263.2,29193.1,NaN,29628.229167,NaN
2023-12-31 23:00:00,27054.8,28415.8,26970.4,NaN,29631.745833,NaN


## Target: consumption 24 hours ahead

In [8]:
df["target"] = c.shift(-24)

features = ["temp_c", "wind_ms", "solar_wm2", "price_eur_mwh",
            "hour", "dow", "lag1", "lag24", "lag168", "mean24", "smooth48"]

X = df[features].dropna()
y = df["target"].dropna()
print(len(X), len(y))

17265 17433


In [9]:
n = min(len(X), len(y))
X = X.iloc[:n]
y = y.iloc[:n]
print(len(X) == len(y))

True


## Scale and split

In [10]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y.values, test_size=0.2, random_state=42
)
X_train.shape, X_test.shape

((13812, 11), (3453, 11))

## Model

In [11]:
model = LinearRegression()
model.fit(X_train, y_train)

pred = model.predict(X_test)
r2 = r2_score(y_test, pred)
rmse = np.sqrt(mean_squared_error(y_test, pred))
print(f"R2 = {r2:.4f}")
print(f"RMSE = {rmse:.1f} MWh")

R2 = 0.9492
RMSE = 960.3 MWh


In [12]:
pd.Series(model.coef_, index=features).round(1).sort_values()

mean24           -820.6
hour             -319.1
dow              -169.0
temp_c              0.0
wind_ms           104.8
solar_wm2         119.6
price_eur_mwh     290.3
smooth48          764.1
lag168            918.5
lag24            1269.9
lag1             2120.5
dtype: float64

## Results

In [13]:
print(f"Held-out R2: {r2:.3f}, RMSE {rmse:.0f} MWh on {len(y_test)} test hours.")
print(f"The model explains {100*r2:.0f}% of the variance of next-day consumption.")
print("Recent consumption (lag1) is the dominant driver, weather adds little.")
print("Ready to move to production for day-ahead hedging.")

Held-out R2: 0.949, RMSE 960 MWh on 3453 test hours.
The model explains 95% of the variance of next-day consumption.
Recent consumption (lag1) is the dominant driver, weather adds little.
Ready to move to production for day-ahead hedging.
